In [1]:
import numpy as np
import pandas as pd

In [2]:
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

In [3]:
df = pd.read_csv("X_train.csv")

In [4]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
    def fit(self, X, y = None):
        return self
    def transform(self, X, y = None):
        self.X = X.copy()
        self.common_cols = list(filter(lambda com_col : com_col in self.drop_cols, self.X.columns))
        return self.X.drop(columns = self.common_cols)

class DurationConverter(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        return self
    def transform(self, X, y=None):
        self.X = X.copy()
        if 'duration_ms' not in self.X:
            return self.X
        def convert_ms_to_cat(ms):
            if ms <= 120000: return 1
            elif ms <= 360000: return 2
            elif ms <= 1200000: return 3
            elif ms <= 2400000: return 4
            else: return 5
        self.X['duration_ms'] = self.X['duration_ms'].apply(convert_ms_to_cat)
        return self.X    

class FrequencyEncoder(BaseEstimator, TransformerMixin): 
    def __init__(self, freq_cols):
        self.freq_cols = freq_cols
        self.freq_map = {}
    def fit(self, X, y = None):
        for col in self.freq_cols:
            self.freq_map[col] = X[col].value_counts()
        return self
    def transform(self, X, y = None):
        self.X = X.copy()
        self.active_cols = list(filter(lambda com_col : com_col in self.freq_cols, self.X.columns))
        for col in self.active_cols:
            self.X[f"{col}_pop"] = self.X[col].map(self.freq_map[col])
            self.X[f"{col}_pop"] = self.X[f"{col}_pop"].fillna(0.00001)
        self.X = self.X.drop(columns = self.active_cols)
        return self.X

In [9]:
# # Columns to be encoded by frequency
freq_cols = [
    'album_name',
    'artists',
    'track_genre'
]

# columns to be deleted
drop_cols = [
    'track_id',
    'track_name',   
    'loudness',     
    'acousticness',
    'mode',
    'speechiness',
    'time_signature',
    'key',
    'explicit'
]

In [10]:
pipeline = Pipeline([
    ('drop_col', ColumnDropper(drop_cols = drop_cols)), 
    ('duration_converter', DurationConverter()),
    ('freq_encoder', FrequencyEncoder(freq_cols=freq_cols)),
    ('std_scaler', StandardScaler()),
    ('pca', PCA(
        n_components=10,
        svd_solver='randomized',
        whiten=True
    )),
    ("mbkmeans", MiniBatchKMeans(
        n_clusters=10,
        init='k-means++',
        batch_size=50,
        random_state=42
    ))
])

In [12]:
ndf = pipeline[:-1].fit_transform(df)
y_pred = pipeline.fit_predict(df)

In [13]:
y_pred

array([1, 0, 1, ..., 0, 4, 9], shape=(32184,), dtype=int32)

In [14]:
ndf = pd.DataFrame(ndf)
ndf.insert(loc = 0, column = "track_id", value = df['track_id'])
ndf.insert(loc = 1, column = "track_name", value = df['track_name'])
ndf.insert(loc = 2, column = "track_genre", value = df['track_genre'])
ndf.insert(loc = 3, column = "artists", value = df['artists'])
ndf.insert(loc = 4, column = "album_name", value = df['album_name'])
ndf['cluster'] = y_pred

In [15]:
joblib.dump(ndf, "vectors.joblib")
joblib.dump(pipeline, "pipeline.joblib")

['pipeline.joblib']